
# XGBoost — pipeline final con Nested Cross-Validation 5×5

Este notebook deja únicamente el flujo final de XGBoost:

1. Carga de datos.
2. Preprocesamiento fijado por variante:
   - MX → `normal`
   - ES → `lemma`
   - CU → `stem`
3. Representación fijada: **TF-IDF de palabras (1,2) + 9 características lingüísticas**.
4. **Nested Cross-Validation 5×5**:
   - Outer CV = 5 folds.
   - Inner CV = 5 folds.
   - Optuna optimiza hiperparámetros únicamente dentro del train de cada outer fold.
5. Optimización final sobre todo el train oficial.
6. Entrenamiento final.
7. Evaluación única sobre el test oficial.

> Nota metodológica: este notebook trata el preprocessing y la representación como decisiones ya fijadas. La Nested CV evalúa la selección de hiperparámetros de XGBoost condicionada a esas decisiones.


In [22]:

# ============================================================
# 1. IMPORTS Y CONFIGURACIÓN GENERAL
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import optuna

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
DATA_DIR = "../data"

VARIANTES = ["mx", "es", "cu"]

FEATURE_COLS = [
    "n_exc",
    "n_int",
    "n_may",
    "n_emo",
    "n_ris",
    "n_neg",
    "n_elo",
    "n_com",
    "n_pun",
]

PREPROCESAMIENTOS = {
    "normal": "",
    "stem": "_stem",
    "lemma": "_lemma",
}

MEJOR_PREP = {
    "mx": "normal",
    "es": "lemma",
    "cu": "stem",
}

# Nested CV
N_OUTER = 5
N_INNER = 5

# Para no disparar el tiempo de cómputo.
# Si luego quieres una corrida más exhaustiva, puedes subirlos.
N_TRIALS_NESTED = 20
N_TRIALS_FINAL = 40


## 2. Carga de datos

In [23]:

def cargar_split(variante, prep, split="train"):
    sufijo = PREPROCESAMIENTOS[prep]
    ruta = f"{DATA_DIR}/{split}_clean{sufijo}_{variante}.csv"
    df = pd.read_csv(ruta)
    return df


def preparar_xy(df):
    columnas = ["MESSAGE_CLEAN"] + FEATURE_COLS

    faltantes = [c for c in columnas + ["IS_IRONIC"] if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")

    X = df[columnas].copy()
    y = df["IS_IRONIC"].astype(int).copy()

    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[FEATURE_COLS] = X[FEATURE_COLS].fillna(0)

    return X, y


## 3. Representación fija: TF-IDF (1,2) + 9 features

In [24]:

def crear_preprocesador():
    tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=20000,
        lowercase=False,
        sublinear_tf=True,
        dtype=np.float32,
    )

    return ColumnTransformer(
        transformers=[
            ("tfidf", tfidf, "MESSAGE_CLEAN"),
            ("linguisticas", "passthrough", FEATURE_COLS),
        ],
        remainder="drop",
    )


## 4. Espacio general de búsqueda para Optuna

In [25]:

# ============================================================
# 4. ESPACIO GENERAL DE BÚSQUEDA PARA OPTUNA
# ============================================================
# Se utiliza el mismo espacio de búsqueda para MX, ES y CU.
# La selección de hiperparámetros se realiza únicamente dentro
# del Inner CV de cada fold externo.

def sugerir_params(trial):
    return {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 800, step=50
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 2, 10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.30, log=True
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.60, 1.00
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.50, 1.00
        ),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 20.0, log=True
        ),
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 1.0, 3.0
        ),
    }


def crear_xgboost(params, random_state=RANDOM_STATE):
    return XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1,
        **params,
    )


def crear_pipeline(params, random_state=RANDOM_STATE):
    return Pipeline(
        steps=[
            ("features", crear_preprocesador()),
            ("xgb", crear_xgboost(params, random_state=random_state)),
        ]
    )



## 5. Nested Cross-Validation 5×5

En cada fold externo:

- 80% del train queda como `outer_train`.
- 20% queda como `outer_validation`.
- Optuna solo ve `outer_train`.
- Cada trial de Optuna se evalúa mediante **Inner Stratified 5-Fold**.
- La mejor configuración se reentrena con todo `outer_train`.
- Se evalúa una sola vez en `outer_validation`.

El promedio de los cinco folds externos es el **F1-Macro Nested CV**.


In [28]:
def nested_cv_variante(variante, n_trials=N_TRIALS_NESTED):
    prep = MEJOR_PREP[variante]

    df = cargar_split(variante, prep, split="train")
    X, y = preparar_xy(df)

    outer_cv = StratifiedKFold(
        n_splits=N_OUTER,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    resultados_outer = []

    for outer_fold, (idx_train, idx_val) in enumerate(
        outer_cv.split(X, y),
        start=1,
    ):
        print(f"\n{'='*70}")
        print(f"{variante.upper()} — OUTER FOLD {outer_fold}/{N_OUTER}")
        print(f"{'='*70}")

        X_outer_train = X.iloc[idx_train].copy()
        y_outer_train = y.iloc[idx_train].copy()

        X_outer_val = X.iloc[idx_val].copy()
        y_outer_val = y.iloc[idx_val].copy()

        inner_cv = StratifiedKFold(
            n_splits=N_INNER,
            shuffle=True,
            random_state=RANDOM_STATE + outer_fold,
        )

        def objective(trial, outer_fold=outer_fold):
            params = sugerir_params(trial)

            pipeline = crear_pipeline(
                params,
                random_state=RANDOM_STATE + outer_fold,
            )

            scores = cross_val_score(
                estimator=pipeline,
                X=X_outer_train,
                y=y_outer_train,
                cv=inner_cv,
                scoring="f1_macro",
                n_jobs=1,
            )

            return scores.mean()

        sampler = optuna.samplers.TPESampler(
            seed=RANDOM_STATE + outer_fold
        )

        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
        )

        study.optimize(
            objective,
            n_trials=n_trials,
            show_progress_bar=True,
        )

        best_params = study.best_params

        modelo_outer = crear_pipeline(
            best_params,
            random_state=RANDOM_STATE + outer_fold,
        )

        modelo_outer.fit(
            X_outer_train,
            y_outer_train,
        )

        pred = modelo_outer.predict(
            X_outer_val
        )

        f1_outer = f1_score(
            y_outer_val,
            pred,
            average="macro",
        )

        resultados_outer.append({
            "variante": variante,
            "outer_fold": outer_fold,
            "inner_best_f1": study.best_value,
            "outer_f1_macro": f1_outer,
            "best_params": best_params,
        })

        print(f"Mejor F1 inner: {study.best_value:.4f}")
        print(f"F1-Macro outer: {f1_outer:.4f}")

    df_outer = pd.DataFrame(resultados_outer)

    resumen = {
        "variante": variante,
        "preprocesamiento": prep,
        "f1_nested_mean": df_outer["outer_f1_macro"].mean(),
        "f1_nested_std": df_outer["outer_f1_macro"].std(ddof=1),
        "f1_nested_min": df_outer["outer_f1_macro"].min(),
        "f1_nested_max": df_outer["outer_f1_macro"].max(),
    }

    return df_outer, resumen

In [29]:

# ============================================================
# 6. EJECUTAR NESTED CV PARA LAS 3 VARIANTES
# ============================================================

nested_detalle = {}
nested_resumen = []

for variante in VARIANTES:
    detalle, resumen = nested_cv_variante(
        variante,
        n_trials=N_TRIALS_NESTED,
    )

    nested_detalle[variante] = detalle
    nested_resumen.append(resumen)

df_nested_resumen = pd.DataFrame(nested_resumen)

print("\nRESULTADOS NESTED CV 5×5")
display(
    df_nested_resumen.style.format({
        "f1_nested_mean": "{:.4f}",
        "f1_nested_std": "{:.4f}",
        "f1_nested_min": "{:.4f}",
        "f1_nested_max": "{:.4f}",
    })
)



MX — OUTER FOLD 1/5


  0%|          | 0/20 [00:00<?, ?it/s]

Mejor F1 inner: 0.6037
F1-Macro outer: 0.5878

MX — OUTER FOLD 2/5


  0%|          | 0/20 [00:00<?, ?it/s]

Mejor F1 inner: 0.6089
F1-Macro outer: 0.6195

MX — OUTER FOLD 3/5


...

  0%|          | 0/20 [00:00<?, ?it/s]

Mejor F1 inner: 0.6574
F1-Macro outer: 0.6510

CU — OUTER FOLD 5/5


  0%|          | 0/20 [00:00<?, ?it/s]

Mejor F1 inner: 0.6675
F1-Macro outer: 0.6456

RESULTADOS NESTED CV 5×5


,variante,preprocesamiento,f1_nested_mean,f1_nested_std,f1_nested_min,f1_nested_max
0,mx,normal,0.6068,0.0173,0.5878,0.6197
1,es,lemma,0.6935,0.0242,0.6688,0.7305
2,cu,stem,0.6482,0.0137,0.6304,0.6684


In [30]:

# Detalle de los 5 outer folds por variante

for variante in VARIANTES:
    print(f"\nDETALLE OUTER — {variante.upper()}")
    display(
        nested_detalle[variante][
            ["outer_fold", "inner_best_f1", "outer_f1_macro"]
        ].style.format({
            "inner_best_f1": "{:.4f}",
            "outer_f1_macro": "{:.4f}",
        })
    )



DETALLE OUTER — MX


,outer_fold,inner_best_f1,outer_f1_macro
0,1,0.6037,0.5878
1,2,0.6089,0.6195
2,3,0.6082,0.6191
3,4,0.6144,0.6197
4,5,0.6090,0.5879



DETALLE OUTER — ES


,outer_fold,inner_best_f1,outer_f1_macro
0,1,0.7140,0.6785
1,2,0.7087,0.6867
2,3,0.7129,0.7032
3,4,0.7020,0.6688
4,5,0.6966,0.7305



DETALLE OUTER — CU


,outer_fold,inner_best_f1,outer_f1_macro
0,1,0.6480,0.6456
1,2,0.6482,0.6684
2,3,0.6699,0.6304
3,4,0.6574,0.6510
4,5,0.6675,0.6456



## 7. Optimización final sobre todo el train

La Nested CV anterior sirve para estimar rendimiento.  
Ahora se vuelve a ejecutar Optuna sobre **todo el train oficial** para obtener los hiperparámetros del modelo que se entrenará finalmente.

El test oficial todavía no participa.


In [31]:

def optimizar_final_variante(variante, n_trials=N_TRIALS_FINAL):
    prep = MEJOR_PREP[variante]

    df = cargar_split(variante, prep, split="train")
    X, y = preparar_xy(df)

    cv_final = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    def objective(trial):
        params = sugerir_params(trial)

        pipeline = crear_pipeline(
            params,
            random_state=RANDOM_STATE,
        )

        scores = cross_val_score(
            estimator=pipeline,
            X=X,
            y=y,
            cv=cv_final,
            scoring="f1_macro",
            n_jobs=1,
        )

        return scores.mean()

    sampler = optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
    )

    return study


studies_finales = {}

for variante in VARIANTES:
    print(f"\n{'='*70}")
    print(f"OPTIMIZACIÓN FINAL — {variante.upper()}")
    print(f"{'='*70}")

    study = optimizar_final_variante(
        variante,
        n_trials=N_TRIALS_FINAL,
    )

    studies_finales[variante] = study

    print(f"Mejor F1-CV: {study.best_value:.4f}")
    print("Mejores hiperparámetros:")
    for k, v in study.best_params.items():
        print(f"  {k}: {repr(v)}")



OPTIMIZACIÓN FINAL — MX


  0%|          | 0/40 [00:00<?, ?it/s]

Mejor F1-CV: 0.6155
Mejores hiperparámetros:
  n_estimators: 650
  max_depth: 4
  learning_rate: 0.06158802642656143
  min_child_weight: 1
  subsample: 0.673158036914205
  colsample_bytree: 0.773249791303957
  gamma: 2.0366958717256205
  reg_alpha: 0.01714096528869879
  reg_lambda: 1.0032532318247116
  scale_pos_weight: 1.7587808404938527

OPTIMIZACIÓN FINAL — ES


  0%|          | 0/40 [00:00<?, ?it/s]

Mejor F1-CV: 0.7132
Mejores hiperparámetros:
  n_estimators: 650
  max_depth: 3
  learning_rate: 0.04112035080491694
  min_child_weight: 1
  subsample: 0.9455520835632656
  colsample_bytree: 0.920555932808492
  gamma: 2.6716980528236336
  reg_alpha: 0.0005300868857454632
  reg_lambda: 1.3302541205524063
  scale_pos_weight: 1.684812358171659

OPTIMIZACIÓN FINAL — CU


  0%|          | 0/40 [00:00<?, ?it/s]

Mejor F1-CV: 0.6684
Mejores hiperparámetros:
  n_estimators: 450
  max_depth: 3
  learning_rate: 0.03154838798956611
  min_child_weight: 1
  subsample: 0.7859201240583127
  colsample_bytree: 0.825427700298568
  gamma: 2.796169463111169
  reg_alpha: 0.013918300303802592
  reg_lambda: 0.28830895088514297
  scale_pos_weight: 1.7041811146914616


In [32]:

# Resumen de hiperparámetros finales

filas = []

for variante, study in studies_finales.items():
    fila = {
        "variante": variante,
        "preprocesamiento": MEJOR_PREP[variante],
        "best_f1_cv_final": study.best_value,
    }
    fila.update(study.best_params)
    filas.append(fila)

df_params_finales = pd.DataFrame(filas)

display(
    df_params_finales.style.format({
        "best_f1_cv_final": "{:.4f}",
        "learning_rate": "{:.6f}",
        "subsample": "{:.4f}",
        "colsample_bytree": "{:.4f}",
        "gamma": "{:.4f}",
        "reg_alpha": "{:.6f}",
        "reg_lambda": "{:.6f}",
        "scale_pos_weight": "{:.4f}",
    })
)


,variante,preprocesamiento,best_f1_cv_final,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,scale_pos_weight
0,mx,normal,0.6155,650,4,0.061588,1,0.6732,0.7732,2.0367,0.017141,1.003253,1.7588
1,es,lemma,0.7132,650,3,0.041120,1,0.9456,0.9206,2.6717,0.000530,1.330254,1.6848
2,cu,stem,0.6684,450,3,0.031548,1,0.7859,0.8254,2.7962,0.013918,0.288309,1.7042



## 8. Guardar resultados antes de tocar el test

Así los resultados de Nested CV y los hiperparámetros finales quedan guardados aunque se cierre VS Code.


In [33]:

os.makedirs(DATA_DIR, exist_ok=True)

df_nested_resumen.to_csv(
    f"{DATA_DIR}/xgboost_nestedcv_resumen.csv",
    index=False,
)

for variante in VARIANTES:
    nested_detalle[variante].drop(columns=["best_params"]).to_csv(
        f"{DATA_DIR}/xgboost_nestedcv_detalle_{variante}.csv",
        index=False,
    )

best_params_json = {
    variante: study.best_params
    for variante, study in studies_finales.items()
}

with open(
    f"{DATA_DIR}/xgboost_best_params_final.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(best_params_json, f, indent=4)

print("Resultados guardados.")


Resultados guardados.



## 9. Entrenamiento final y evaluación en test oficial

Ejecuta esta sección **solo cuando la Nested CV y la optimización final estén cerradas**.

Cada modelo:
- se entrena con todo el `train` oficial;
- utiliza threshold estándar `0.50`;
- se evalúa una sola vez en el `test` oficial.


In [34]:

def evaluar_test_final(variante):
    prep = MEJOR_PREP[variante]

    train = cargar_split(variante, prep, split="train")
    test = cargar_split(variante, prep, split="test")

    X_train, y_train = preparar_xy(train)
    X_test, y_test = preparar_xy(test)

    best_params = studies_finales[variante].best_params

    pipeline = crear_pipeline(
        best_params,
        random_state=RANDOM_STATE,
    )

    pipeline.fit(
        X_train,
        y_train,
    )

    y_pred = pipeline.predict(
        X_test
    )

    resultado = {
        "variante": variante,
        "preprocesamiento": prep,
        "f1_macro_test": f1_score(
            y_test, y_pred, average="macro"
        ),
        "accuracy_test": accuracy_score(
            y_test, y_pred
        ),
        "precision_macro_test": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "recall_macro_test": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
    }

    return resultado, pipeline, y_test, y_pred


In [35]:

resultados_test = []
modelos_finales = {}

for variante in VARIANTES:
    print(f"\n{'='*70}")
    print(f"TEST OFICIAL — {variante.upper()}")
    print(f"{'='*70}")

    resultado, pipeline, y_test, y_pred = evaluar_test_final(
        variante
    )

    resultados_test.append(resultado)
    modelos_finales[variante] = pipeline

    print(f"F1-Macro: {resultado['f1_macro_test']:.4f}")
    print(f"Accuracy: {resultado['accuracy_test']:.4f}")
    print(f"Precision Macro: {resultado['precision_macro_test']:.4f}")
    print(f"Recall Macro: {resultado['recall_macro_test']:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_pred,
            digits=4,
            zero_division=0,
        )
    )

    print("Matriz de confusión:")
    print(confusion_matrix(y_test, y_pred))


df_resultados_test = pd.DataFrame(resultados_test)

print("\nRESULTADOS FINALES XGBOOST — TEST OFICIAL")
display(
    df_resultados_test.style.format({
        "f1_macro_test": "{:.4f}",
        "accuracy_test": "{:.4f}",
        "precision_macro_test": "{:.4f}",
        "recall_macro_test": "{:.4f}",
    })
)



TEST OFICIAL — MX
F1-Macro: 0.6176
Accuracy: 0.6517
Precision Macro: 0.6157
Recall Macro: 0.6217

Classification report:
              precision    recall  f1-score   support

           0     0.7540    0.7107    0.7317       401
           1     0.4775    0.5327    0.5036       199

    accuracy                         0.6517       600
   macro avg     0.6157    0.6217    0.6176       600
weighted avg     0.6623    0.6517    0.6560       600

Matriz de confusión:
[[285 116]
 [ 93 106]]

TEST OFICIAL — ES
F1-Macro: 0.6875
Accuracy: 0.7117
Precision Macro: 0.6836
Recall Macro: 0.6963

Classification report:
              precision    recall  f1-score   support

           0     0.8093    0.7425    0.7744       400
           1     0.5579    0.6500    0.6005       200

    accuracy                         0.7117       600
   macro avg     0.6836    0.6963    0.6875       600
weighted avg     0.7255    0.7117    0.7165       600

Matriz de confusión:
[[297 103]
 [ 70 130]]

TEST OFICIAL 

,variante,preprocesamiento,f1_macro_test,accuracy_test,precision_macro_test,recall_macro_test
0,mx,normal,0.6176,0.6517,0.6157,0.6217
1,es,lemma,0.6875,0.7117,0.6836,0.6963
2,cu,stem,0.6547,0.7050,0.6636,0.6500


In [36]:

# Guardar resultados finales del test

df_resultados_test.to_csv(
    f"{DATA_DIR}/xgboost_test_final.csv",
    index=False,
)

print(
    f"Guardado en {DATA_DIR}/xgboost_test_final.csv"
)


Guardado en ../data/xgboost_test_final.csv
